# Notebook 05b — Global-Bus LightGBM with Weather Features

## Purpose

This notebook is the global-bus counterpart to notebook 04b: take the global-bus LightGBM pipeline from notebook 05a and augment it with the weather features from notebook 02w. Together with notebook 04b, this completes the 2×2 feature-ablation matrix:

| Architecture / Weather | No weather | With weather |
|---|---|---|
| Zone-direct + top-down | Notebook 04 | Notebook 04b |
| Global-bus direct | Notebook 05a | **Notebook 05b** |

This matrix is the substantive contribution of the weather-augmentation experiment. Notebook 06's evaluation will compare all four cells to answer two coupled research questions:

1. **Does adding weather features improve forecasts?** (compare each column pair)
2. **Which architecture benefits more from weather?** (compare the within-row improvements)

This notebook addresses Q1 specifically for the global-bus architecture, and contributes the bottom-right cell of the matrix.

## Scope of this notebook

This notebook does NOT:
- Train global-bus models from scratch (we warm-start with notebook 05a's hyperparameters)
- Run an Optuna hyperparameter search
- Re-discover `n_estimators` via early-stopping refit (we use notebook 05a's trial-log median scaled by the train-set-size ratio)
- Compute evaluation metrics (notebook 06 handles all evaluation)

This notebook DOES:
- Load notebook 02's bus-level feature parquets for both tasks
- Load notebook 02w's weather features parquet and join via `(zone_name, timestamp)` onto each bus row (so every bus in a given zone-hour gets the same weather)
- For each of 2 task models, load notebook 05a's best hyperparameters from JSON
- Refit on 2022-2024 (train + val combined) with the weather-augmented feature set
- Predict on 2025 at bus level
- Write two forecast parquets in the canonical 7-column schema

## Methodological decisions (locked in)

### Decision 1: Strict warm-start (no Optuna re-tuning, no fresh n_estimators discovery)

We use notebook 05a's already-discovered hyperparameters verbatim, and we set the final-train `n_estimators` to the median of notebook 05a's per-trial `best_iteration` values scaled by `n_finaltrain / n_train ≈ 1.503` — identical to notebook 05a's own final-train logic. **Rationale:**

The methodological purpose of notebook 05b is to isolate the contribution of weather features to the global-bus architecture. If we also re-tune hyperparameters or rediscover the tree count, we confound multiple effects. A strict warm-start lets us attribute any RMSE delta cleanly to the weather features alone.

The decision not to run a fresh discovery refit deserves explicit justification: at bus level, a discovery refit on 65M training rows would itself cost 10-30 minutes per task, comparable to the final-train. The alternative — using notebook 05a's recorded median best_iteration scaled by the same factor — uses information we already have without rerunning anything. This is the same trade-off notebook 05a made for its own final-train.

**Honest limitation:** notebook 05a's hyperparameters and `best_iteration` distribution were discovered against a feature set without weather. Adding 5 strongly-predictive features changes what's optimal — as we saw in notebook 04b, the post-weather optimal `best_iteration` ranged from 31 to 760 across zone-task pairs, often very different from the no-weather optimum. Notebook 05b's models may therefore be undertrained or overtrained for the weather-augmented feature set. We accept this cost in exchange for the controlled-experiment cleanliness; notebook 06 will report what the trade-off actually produces.

### Decision 2: Weather features joined at bus row, not aggregated

The weather features in `weather_features.parquet` are at `(zone_name, timestamp)` granularity. When we join them onto the bus-level feature matrix, every bus within a given zone-hour gets the same 5 weather columns. This is consistent with the structural assumption that weather is a zone-level signal (one weather observation per zone per hour), and it matches how notebook 04b uses these features. The model can use weather as a zone-wide context signal while still differentiating bus-level patterns via `bus_unique_id` as a categorical feature.

### Decision 3: Output filename uses `_weather_` suffix

We name the output files:
- `forecast_global_bus_lgbm_weather_nextday.parquet`
- `forecast_global_bus_lgbm_weather_nextmonth.parquet`

The `_weather_` suffix mirrors notebook 04b's convention and distinguishes these forecasts from notebook 05a's baseline outputs. The `model_name` field within the parquet will be `global_bus_lgbm_weather_{nextday|nextmonth}`.

## How this compares to notebook 05a

| Property | Notebook 05a | Notebook 05b |
|---|---|---|
| Number of models | 2 (1 per task) | 2 (1 per task) |
| Hyperparameters | Discovered via Optuna (10 trials per task) | Same as notebook 05a — loaded from JSON |
| Feature set | Bus + zone-level features (~26-29 features per task) | Same + 5 weather features (~31-34 features per task) |
| Final training | 2022-2024 with median best_iteration × 1.503 | Same |
| Cold-start handling | Implicit via LightGBM unseen-category routing | Same |
| Optuna trials | 20 (10 × 2) | 0 |
| Approx compute | 2.5 hours | **~30-60 min** (no Optuna) |

The structural diff between notebooks 05a and 05b is the addition of 5 weather columns to the X matrices. Everything else is byte-identical in spirit.

## How this compares to notebook 04b

| Property | Notebook 04b (zone-direct + weather) | Notebook 05b (global-bus + weather) |
|---|---|---|
| Number of models | 16 (8 zones × 2 tasks) | 2 (1 per task) |
| Training rows per model | ~17K-26K (zone-aggregated) | ~65M-98M (full bus-level) |
| Target | Zone-aggregated pd | Bus-level pd directly |
| Bus identity in model | Implicit (shares applied post-hoc) | Explicit (categorical feature) |
| Disaggregation step | Yes (hour-of-day shares) | No (direct bus predictions) |
| Cold-start treatment | Share-fallback (explicit) | Categorical-routing (implicit) |
| n_estimators source | Fresh discovery refit | Notebook 05a's median trial × 1.503 |
| Approx compute | ~5 min | **~30-60 min** |

The two notebooks (04b and 05b) form a complementary pair that lets notebook 06 ask: **which architecture better exploits weather features?** A priori, zone-direct may benefit more because zone-level weather is directly aligned with the zone-level target. Global-bus may benefit less from weather but more from bus-specific identity. We don't know the answer in advance.

## Outputs

Two forecast parquet files written to `data/processed/forecasts/`:

| File | Task | model_name |
|---|---|---|
| `forecast_global_bus_lgbm_weather_nextday.parquet` | Next-day | `global_bus_lgbm_weather_nextday` |
| `forecast_global_bus_lgbm_weather_nextmonth.parquet` | Next-month | `global_bus_lgbm_weather_nextmonth` |

Each file: 32,427,554 rows in the 7-column required schema, structurally identical to notebooks 03, 04, 04b, 05a's outputs for row-aligned comparison in notebook 06.

## Runtime estimate

Approximately 30-60 minutes total — much faster than notebook 05a because we skip Optuna entirely. The main cost is the final-train per task on 98M rows.

| Stage | Time |
|---|---|
| Load bus-level features (per task, with weather join) | 30-60 s |
| Build LightGBM Datasets | 30-60 s |
| Final retraining (98M rows, per task) | 10-25 min |
| Prediction (32M rows, per task) | 1-3 min |
| File writing + verification | 30-60 s |
| **Total** | **~30-60 min** |

In [5]:
"""
Imports, paths, and configuration for notebook 05b (global-bus LightGBM + weather).

Loads the same scientific stack as notebook 05a, plus reads the artifacts produced
by notebooks 02, 02w, and 05a from their canonical locations:

  Inputs:
    - data/processed/features/features_{task}_{year}.parquet         (notebook 02)
    - data/processed/weather_features/weather_features.parquet       (notebook 02w)
    - data/processed/model_params/best_params_global_bus_{task}.json (notebook 05a)
    - data/processed/audit/forecastable_bus_list.parquet             (notebook 01)

  Outputs:
    - data/processed/forecasts/forecast_global_bus_lgbm_weather_nextday.parquet
    - data/processed/forecasts/forecast_global_bus_lgbm_weather_nextmonth.parquet

If lightgbm is not installed, the cell fails with a clear install instruction.

Runtime: <1 second.
"""

# Standard library
from pathlib import Path
import warnings
import gc
import time
import json
import psutil

# Numeric and data
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# ML
try:
    import lightgbm as lgb
except ImportError as e:
    raise ImportError(
        "lightgbm is required for notebook 05b. Install with: pip install lightgbm. "
        "On macOS, if the install succeeds but import fails with an OpenMP error, "
        "run: brew install libomp"
    ) from e

# Display and warning configuration
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
warnings.simplefilter("ignore", category=FutureWarning)

# ──────────────────────────────────────────────────────────────────────────
# Paths (relative to notebook location: assignment2/notebooks/)
# ──────────────────────────────────────────────────────────────────────────
DATA_DIR = Path("../data")
AUDIT_DIR = Path("../data/processed/audit")
FEATURES_DIR = Path("../data/processed/features")
WEATHER_FEATURES_DIR = Path("../data/processed/weather_features")
MODEL_PARAMS_DIR = Path("../data/processed/model_params")
FORECASTS_DIR = Path("../data/processed/forecasts")

# Output directory already exists from prior notebooks; ensure for safety
FORECASTS_DIR.mkdir(parents=True, exist_ok=True)

# ──────────────────────────────────────────────────────────────────────────
# Input file paths
# ──────────────────────────────────────────────────────────────────────────
YEARS = [2022, 2023, 2024, 2025]

NEXTDAY_FEATURE_FILES = {y: FEATURES_DIR / f"features_nextday_{y}.parquet" for y in YEARS}
NEXTMONTH_FEATURE_FILES = {y: FEATURES_DIR / f"features_nextmonth_{y}.parquet" for y in YEARS}
WEATHER_FEATURES_PATH = WEATHER_FEATURES_DIR / "weather_features.parquet"
FORECASTABLE_BUS_LIST_PATH = AUDIT_DIR / "forecastable_bus_list.parquet"

# Hyperparameter JSON paths (from notebook 05a)
HYPERPARAM_PATHS = {
    "nextday":   MODEL_PARAMS_DIR / "best_params_global_bus_nextday.json",
    "nextmonth": MODEL_PARAMS_DIR / "best_params_global_bus_nextmonth.json",
}

# Verify all expected inputs exist before proceeding
for y in YEARS:
    assert NEXTDAY_FEATURE_FILES[y].exists(), f"Missing: {NEXTDAY_FEATURE_FILES[y]}"
    assert NEXTMONTH_FEATURE_FILES[y].exists(), f"Missing: {NEXTMONTH_FEATURE_FILES[y]}"
assert WEATHER_FEATURES_PATH.exists(), (
    f"Missing weather features: {WEATHER_FEATURES_PATH}. Run notebook 02w first."
)
assert FORECASTABLE_BUS_LIST_PATH.exists(), (
    f"Missing audit: {FORECASTABLE_BUS_LIST_PATH}. Run notebook 01 first."
)
for task, path in HYPERPARAM_PATHS.items():
    assert path.exists(), f"Missing notebook 05a hyperparameter file: {path}. Run notebook 05a first."

# ──────────────────────────────────────────────────────────────────────────
# Configuration constants (matching notebook 05a)
# ──────────────────────────────────────────────────────────────────────────
TRAIN_YEARS = [2022, 2023]
VAL_YEAR = 2024
FINAL_TRAIN_YEARS = [2022, 2023, 2024]
TEST_YEAR = 2025
TASKS = ["nextday", "nextmonth"]

# LightGBM seed for reproducibility (matches notebook 05a's OPTUNA_SEED)
LGBM_SEED = 42

# Categorical features for LightGBM (same as notebook 05a)
CATEGORICAL_FEATURES = ["bus_unique_id", "zone_name"]

# Columns to exclude from model features
NON_FEATURE_COLUMNS = ["timestamp", "pd", "is_test_period"]

# Per-task column drops (matching notebook 05a Cell 3 and notebook 04b)
NEXTMONTH_DROP_COLS = ["pd_lag_17520h"]

# Weather feature column names (from notebook 02w's output schema)
WEATHER_FEATURE_COLS = [
    "temp_at_hour",
    "HDH_at_hour",
    "CDH_at_hour",
    "temp_trailing_24h_at_fc",
    "temp_trailing_168h_at_fc",
]

# Output file paths
OUTPUT_PATHS = {
    "nextday":   FORECASTS_DIR / "forecast_global_bus_lgbm_weather_nextday.parquet",
    "nextmonth": FORECASTS_DIR / "forecast_global_bus_lgbm_weather_nextmonth.parquet",
}

# Model name convention for the parquet model_name field
MODEL_NAMES = {
    "nextday":   "global_bus_lgbm_weather_nextday",
    "nextmonth": "global_bus_lgbm_weather_nextmonth",
}

# ──────────────────────────────────────────────────────────────────────────
# Load notebook 05a hyperparameters and compute n_estimators per task
# ──────────────────────────────────────────────────────────────────────────
# For each task we need:
#   - best_params: 9 LightGBM hyperparameters from notebook 05a's Optuna search
#   - best_iters:  per-trial best_iteration values from notebook 05a's trial_log
#   - n_finaltrain_scaling: median(best_iters) × scaling factor
#
# Scaling factor = n_finaltrain / n_train = 98,552,404 / 65,590,110 ≈ 1.503
# Matches notebook 05a's own final-train scaling exactly.
N_TRAIN_ROWS = 65_590_110       # 2022 + 2023 row count (verified in notebook 05a)
N_FINALTRAIN_ROWS = 98_552_404  # 2022 + 2023 + 2024 row count
SCALING_FACTOR = N_FINALTRAIN_ROWS / N_TRAIN_ROWS  # ≈ 1.503

print(f"Loading notebook 05a hyperparameters and computing n_estimators per task:")
print(f"  Scaling factor (n_finaltrain / n_train): {SCALING_FACTOR:.4f}")

task_hyperparams = {}  # {task: {"best_params": dict, "n_estimators_final": int, ...}}

for task, path in HYPERPARAM_PATHS.items():
    with open(path, "r") as f:
        cached = json.load(f)

    best_params = cached["best_params"]
    trial_log = cached["trial_log"]

    # Extract per-trial best_iteration values
    best_iters = [t["best_iteration"] for t in trial_log if t.get("best_iteration", 0) > 0]
    if len(best_iters) == 0:
        raise RuntimeError(f"{task}: no valid best_iteration values in trial_log")

    median_iter = int(np.median(best_iters))
    n_estimators_final = int(round(median_iter * SCALING_FACTOR))

    task_hyperparams[task] = {
        "best_params": best_params,
        "best_iters_optuna": best_iters,
        "median_iter_optuna": median_iter,
        "n_estimators_final": n_estimators_final,
        "no_weather_val_rmse": cached["best_rmse"],
        "n_features_no_weather": cached["n_features"],
    }

    print(f"\n  {task}:")
    print(f"    Optuna best_iters across {len(best_iters)} trials: {best_iters}")
    print(f"    Median: {median_iter}")
    print(f"    Final n_estimators ({median_iter} × {SCALING_FACTOR:.3f}): {n_estimators_final}")
    print(f"    Notebook 05a val RMSE (no weather): {cached['best_rmse']:.4f}")
    print(f"    Hyperparameters: {best_params}")

# ──────────────────────────────────────────────────────────────────────────
# Verify weather features schema
# ──────────────────────────────────────────────────────────────────────────
weather_schema = pq.read_schema(WEATHER_FEATURES_PATH)
weather_cols = [field.name for field in weather_schema]
expected_weather_cols = ["zone_name", "timestamp"] + WEATHER_FEATURE_COLS
assert set(weather_cols) == set(expected_weather_cols), (
    f"Weather features schema mismatch.\n"
    f"  Expected: {sorted(expected_weather_cols)}\n"
    f"  Got:      {sorted(weather_cols)}"
)

# ──────────────────────────────────────────────────────────────────────────
# Print configuration summary
# ──────────────────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"Notebook 05b configuration:")
print(f"{'='*70}")
print(f"  LightGBM version: {lgb.__version__}")
print(f"  Train/val/final-train/test years: {TRAIN_YEARS} / {VAL_YEAR} / {FINAL_TRAIN_YEARS} / {TEST_YEAR}")
print(f"  LightGBM seed: {LGBM_SEED}")
print(f"  Categorical features: {CATEGORICAL_FEATURES}")
print(f"  Weather feature columns ({len(WEATHER_FEATURE_COLS)}): {WEATHER_FEATURE_COLS}")
print(f"  Drop from next-month: {NEXTMONTH_DROP_COLS}")

print(f"\nFinal n_estimators (warm-started from notebook 05a):")
for task, hp in task_hyperparams.items():
    print(f"  {task}: {hp['n_estimators_final']}")

print(f"\nInput paths:")
print(f"  Features dir:        {FEATURES_DIR.resolve()}")
print(f"  Weather features:    {WEATHER_FEATURES_PATH.resolve()}")
print(f"  Hyperparameter JSONs: {MODEL_PARAMS_DIR.resolve()}")

print(f"\nOutput paths:")
for task, path in OUTPUT_PATHS.items():
    exists_str = " (already exists — will overwrite)" if path.exists() else ""
    print(f"  {task}: {path.name}{exists_str}")

mem = psutil.virtual_memory()
print(f"\nSystem RAM available: {mem.available / 1024**3:.1f} GB / {mem.total / 1024**3:.1f} GB total")
print(f"\n✓ All inputs verified. Ready to load per-task data in Cell 3.")

Loading notebook 05a hyperparameters and computing n_estimators per task:
  Scaling factor (n_finaltrain / n_train): 1.5025

  nextday:
    Optuna best_iters across 10 trials: [23, 663, 282, 78, 378, 225, 27, 217, 45, 294]
    Median: 221
    Final n_estimators (221 × 1.503): 332
    Notebook 05a val RMSE (no weather): 5.3296
    Hyperparameters: {'num_leaves': 202, 'learning_rate': 0.026000059117302653, 'min_data_in_leaf': 2759, 'feature_fraction': 0.6563696899899051, 'bagging_fraction': 0.9208787923016158, 'bagging_freq': 1, 'lambda_l1': 7.6204817861585425, 'lambda_l2': 0.08916674715636552, 'cat_smooth': 20}

  nextmonth:
    Optuna best_iters across 10 trials: [10, 359, 125, 42, 239, 152, 18, 158, 43, 126]
    Median: 125
    Final n_estimators (125 × 1.503): 188
    Notebook 05a val RMSE (no weather): 7.6688
    Hyperparameters: {'num_leaves': 371, 'learning_rate': 0.010725209743171997, 'min_data_in_leaf': 4853, 'feature_fraction': 0.9329770563201687, 'bagging_fraction': 0.68493564

### Configuration verified — observations

All inputs are present and verified. Both hyperparameter JSON files from notebook 05a loaded cleanly, and the median-best-iteration calculation reproduces notebook 05a's own final-train tree counts exactly:

| Task | Median Optuna `best_iteration` | Scaling factor | Final n_estimators |
|---|---|---|---|
| nextday | 221 | 1.5025 | **332** |
| nextmonth | 125 | 1.5025 | **188** |

These values match what notebook 05a actually used during its own final-train phase (verified in the previous session's PDF check: nextday final-train ran 332 trees, nextmonth ran 188). The reproducibility confirms we're inheriting notebook 05a's tree-count logic exactly, not just approximating it.

**The Optuna `best_iteration` distributions are wide.** For nextday: trial best_iters ranged from 23 (fastest early-stopping) to 663 (deepest training). The median of 221 sits roughly in the middle. For nextmonth: range 10 to 359, median 125. The wide ranges reflect Optuna's hyperparameter search exploring very different learning-rate / num_leaves combinations — some converged in 2-3 minutes with shallow trees, others ran for 25+ minutes with deeper trees. The median is a robust central tendency measure that's less sensitive to these extremes than the mean would be.

**Notebook 05a's val RMSEs (no weather) provide reference points for later comparison:**
- nextday: 5.3296 MW
- nextmonth: 7.6688 MW

These are the values to beat at the val stage. We don't compute notebook 05b's val RMSE during this notebook (we skip the discovery refit), so the direct comparison happens in notebook 06 at test time using 2025 actuals.

**Hyperparameter highlights worth noting:**
- nextday: `num_leaves=202`, `learning_rate=0.026`, `min_data_in_leaf=2759`, `cat_smooth=20`
- nextmonth: `num_leaves=371`, `learning_rate=0.011`, `min_data_in_leaf=4853`, `cat_smooth=53`

The very large `min_data_in_leaf` values (2,759 and 4,853) are notebook 05a's signal that bus-level pd has high noise — each leaf needs thousands of training rows of support before LightGBM is willing to commit to a split. This carries over to notebook 05b unchanged. The `cat_smooth` values control how much the 4,208-bus categorical feature gets regularized toward zone-level means..

## Per-task processing loop (load → join weather → final-train → predict → write)

Cell 3 processes the two forecasting tasks sequentially in a single loop. For each task we:

1. **Load** the four yearly bus-level feature parquets for that task and concatenate.
2. **Join weather features** on `(zone_name, timestamp)` so every bus row in a zone-hour gets the same 5 weather columns.
3. **Slice** into train (2022-2023), val (2024), final-train (2022-2024), and test (2025) sets.
4. **Build** LightGBM Datasets with `free_raw_data=True` (releases pandas after binning).
5. **Final-train** a fresh LightGBM model on 2022-2024 using notebook 05a's hyperparameters and the precomputed `n_estimators` from Cell 2.
6. **Predict** on the 2025 test set.
7. **Write** the forecast parquet in the canonical 7-column schema.
8. **Release** all task-specific data before moving to the next task.

**Per-task isolation is the central memory-management design.** The previous load-everything-upfront approach (initial notebook 05a draft) triggered macOS memory compression at peak; the per-task pattern we adopted in notebook 05a's final design keeps peak memory at ~18-22 GB per task and recovers to ~3-4 GB baseline between tasks. We follow the same pattern here.

**No Optuna search, no discovery refit, no validation phase.** The strict warm-start design means we skip everything notebook 05a did before its own final-train step. Each task goes directly from data loading to final-train using the precomputed `n_estimators` from Cell 2. The structural difference from notebook 05a's per-task loop is approximately: skip steps [3] Optuna search and [3.5] discovery refit, jump directly to [4] final retraining with the precomputed tree count.

**Checkpoint recovery via output-file presence.** If a forecast file already exists on disk for a task (e.g., the kernel died mid-second-task and we re-run the cell), that task is skipped. The other task — whose forecast file doesn't yet exist — is processed normally. This is simpler than the JSON-checkpoint approach we used in notebook 05a because there's no Optuna state to preserve; the only persistent artifact per task is the forecast file itself.

**Expected memory profile per task:**
- After data load: ~14 GB peak (full bus-level DataFrame in pandas)
- After Dataset construction with `free_raw_data=True`: ~7 GB (LightGBM binned representation only)
- During training: ~9-11 GB (binned data + model state + working memory)
- After prediction + write: ~3-4 GB (output DataFrame still in memory until released)
- After explicit `del` + `gc.collect()`: ~3-4 GB baseline

**Expected runtime per task:**
- Load + join: 30-60 s
- Build Datasets: 30-60 s
- Final-train: 10-25 min (the dominant cost)
- Predict: 1-3 min
- Format + write: 30-60 s
- **Per-task total: 13-30 min**

Total for both tasks: 26-60 minutes. The kernel should be left undisturbed during the final-train phases.

In [6]:
"""
Per-task processing loop for global-bus LightGBM with weather features.

For each task in [nextday, nextmonth]:
  1. Skip if the output forecast parquet already exists (checkpoint recovery).
  2. Load the 4 yearly bus-level feature parquets, applying task-specific column drops.
  3. Join weather features from notebook 02w on (zone_name, timestamp).
  4. Slice into train/val/finaltrain/test by year.
  5. Build LightGBM Datasets for finaltrain with free_raw_data=True.
  6. Refit LightGBM using notebook 05a's hyperparameters and the precomputed
     n_estimators from Cell 2.
  7. Predict on the 2025 test set.
  8. Format into the 7-column canonical schema and write the parquet.
  9. Release all task-specific data before the next task.

Mirrors notebook 05a Cell 5 minus the Optuna search and discovery refit. The
n_estimators value (332 for nextday, 188 for nextmonth) comes from task_hyperparams
which we computed in Cell 2 as median(trial best_iter) × 1.503.

Memory: peak ~18-22 GB per task; ~3-4 GB baseline between tasks.
Runtime: ~13-30 min per task; 26-60 min total.
"""

t0_outer = time.time()


# ──────────────────────────────────────────────────────────────────────────
# Helper: load one task's feature data and join weather features
# ──────────────────────────────────────────────────────────────────────────
def load_task_data_with_weather(task):
    """
    Load all 4 yearly bus-level feature parquets for a task, apply column drops,
    concatenate, and join weather features on (zone_name, timestamp).

    Returns a single DataFrame ready for splits + Dataset construction.

    Memory: peak ~14 GB during concat; ~13 GB after the join.
    """
    t_load = time.time()
    files_dict = NEXTDAY_FEATURE_FILES if task == "nextday" else NEXTMONTH_FEATURE_FILES
    drop_cols = NEXTMONTH_DROP_COLS if task == "nextmonth" else []

    print(f"  Loading {task} feature parquets and joining weather...")

    # Load 4 yearly parquets
    year_dfs = []
    for y in YEARS:
        df = pq.read_table(files_dict[y]).to_pandas()
        if drop_cols:
            df = df.drop(columns=[c for c in drop_cols if c in df.columns])
        year_dfs.append(df)
    bus_df = pd.concat(year_dfs, ignore_index=True)
    del year_dfs
    gc.collect()

    mem_after_load = bus_df.memory_usage(deep=True).sum() / 1024**3
    elapsed_load = time.time() - t_load
    print(f"    Loaded: {len(bus_df):,} rows × {bus_df.shape[1]} cols "
          f"({mem_after_load:.2f} GB) in {elapsed_load:.1f}s")

    # Load weather features
    t_weather = time.time()
    weather_df = pd.read_parquet(WEATHER_FEATURES_PATH)

    # Reconcile dtypes for the join keys
    if weather_df["timestamp"].dtype != bus_df["timestamp"].dtype:
        weather_df["timestamp"] = weather_df["timestamp"].astype(bus_df["timestamp"].dtype)

    # zone_name in bus_df is categorical; in weather_df it's also categorical but
    # the category orderings might differ. Cast both to string for safe merge,
    # then re-cast bus_df's joined zone_name back to category afterwards.
    bus_df["zone_name"] = bus_df["zone_name"].astype(str)
    weather_df["zone_name"] = weather_df["zone_name"].astype(str)

    # Left-join weather
    n_pre = len(bus_df)
    bus_df = bus_df.merge(weather_df, on=["zone_name", "timestamp"], how="left")
    n_post = len(bus_df)
    assert n_post == n_pre, f"Join changed row count: {n_pre:,} → {n_post:,}"

    # Restore categorical dtype for the LightGBM categorical handling
    bus_df["zone_name"] = bus_df["zone_name"].astype("category")

    # Count NaN in weather columns (expect ~few thousand from DST gaps × 4,208 buses)
    n_nan_weather = bus_df[WEATHER_FEATURE_COLS].isna().any(axis=1).sum()
    pct_nan = 100 * n_nan_weather / n_post

    elapsed_weather = time.time() - t_weather
    mem_after_join = bus_df.memory_usage(deep=True).sum() / 1024**3
    print(f"    Joined weather in {elapsed_weather:.1f}s")
    print(f"    Rows with NaN in any weather col: {n_nan_weather:,} ({pct_nan:.3f}%)")
    print(f"    Post-join: {len(bus_df):,} rows × {bus_df.shape[1]} cols ({mem_after_join:.2f} GB)")

    return bus_df


# ──────────────────────────────────────────────────────────────────────────
# Helper: slice into train/val/finaltrain/test for a task DataFrame
# ──────────────────────────────────────────────────────────────────────────
def build_splits(bus_df):
    """
    Given a task DataFrame with weather features joined, return slices:
      X_finaltrain, y_finaltrain  (2022-2024, for final training)
      X_test, y_test              (2025, for prediction)
      test_identity               (bus_unique_id, zone_name, timestamp for test rows)

    We do not return train/val separately because notebook 05b skips the
    discovery refit. Only finaltrain (for fitting) and test (for predicting) are needed.

    Feature columns: all columns except NON_FEATURE_COLUMNS.
    """
    feature_cols = [c for c in bus_df.columns if c not in NON_FEATURE_COLUMNS]
    years = bus_df["timestamp"].dt.year

    finaltrain_mask = years.isin(FINAL_TRAIN_YEARS)
    test_mask = years == TEST_YEAR

    splits = {
        "X_finaltrain":  bus_df.loc[finaltrain_mask, feature_cols].copy(),
        "y_finaltrain":  bus_df.loc[finaltrain_mask, "pd"].copy(),
        "X_test":        bus_df.loc[test_mask, feature_cols].copy(),
        "y_test":        bus_df.loc[test_mask, "pd"].copy(),
        "test_identity": bus_df.loc[test_mask, ["bus_unique_id", "zone_name", "timestamp"]].copy(),
        "feature_cols": feature_cols,
    }
    return splits


# ──────────────────────────────────────────────────────────────────────────
# Helper: format predictions into 7-column output schema
# ──────────────────────────────────────────────────────────────────────────
def format_output(task, predict_pd_arr, test_identity):
    """
    Build the 7-column output DataFrame in the canonical schema.

    Mirrors notebook 05a Cell 5's predict_and_write_for_task formatting,
    including the year/month decomposition pattern for nextmonth's
    forecast_created_at.
    """
    target_date = test_identity["timestamp"].dt.normalize()
    he = (test_identity["timestamp"].dt.hour + 1).astype("int8")

    if task == "nextday":
        forecast_created_at = target_date - pd.Timedelta(days=1)
    else:  # nextmonth: first of (target_month - 1)
        target_year_s = test_identity["timestamp"].dt.year
        target_month_s = test_identity["timestamp"].dt.month
        prev_month_s = target_month_s - 1
        prev_year_s = target_year_s.where(prev_month_s >= 1, target_year_s - 1)
        prev_month_s = prev_month_s.where(prev_month_s >= 1, 12)
        forecast_created_at = pd.to_datetime(
            pd.DataFrame({"year": prev_year_s, "month": prev_month_s, "day": 1})
        )

    output_df = pd.DataFrame({
        "model_name": MODEL_NAMES[task],
        "forecast_created_at": forecast_created_at.values,
        "target_date": target_date.values,
        "he": he.values,
        "bus_id": test_identity["bus_unique_id"].values,
        "zone_id": test_identity["zone_name"].values,
        "predict_pd": predict_pd_arr.astype("float32"),
    })

    return output_df


# ──────────────────────────────────────────────────────────────────────────
# Main per-task loop
# ──────────────────────────────────────────────────────────────────────────
processing_summary = {}

for task_idx, task in enumerate(TASKS, start=1):
    print(f"\n{'='*80}")
    print(f"[{task_idx}/2] Processing task: {task}")
    print(f"{'='*80}")
    t_task = time.time()

    # Skip if output forecast file already exists (checkpoint recovery)
    forecast_path = OUTPUT_PATHS[task]
    if forecast_path.exists():
        size_mb = forecast_path.stat().st_size / 1024**2
        print(f"  Output forecast file already exists ({size_mb:.1f} MB). Skipping task.")
        processing_summary[task] = {"skipped": True, "elapsed_min": 0}
        continue

    # Step 1: Load data with weather join
    print(f"\n  [1/5] Loading task data with weather features...")
    bus_df = load_task_data_with_weather(task)
    mem = psutil.virtual_memory()
    print(f"  RAM available after load: {mem.available / 1024**3:.1f} GB")

    # Step 2: Build splits
    print(f"\n  [2/5] Building train/test splits...")
    t_split = time.time()
    splits = build_splits(bus_df)
    feature_cols = splits["feature_cols"]
    weather_in_features = [c for c in feature_cols if c in WEATHER_FEATURE_COLS]
    print(f"    Final-train: {len(splits['y_finaltrain']):,} rows × {len(feature_cols)} features")
    print(f"    Test:        {len(splits['y_test']):,} rows × {len(feature_cols)} features")
    print(f"    Weather features included: {len(weather_in_features)} ({weather_in_features})")
    print(f"    Total features: {len(feature_cols)} (categoricals: {CATEGORICAL_FEATURES})")
    print(f"    Build splits in {time.time() - t_split:.1f}s")

    # Release the parent DataFrame; splits hold their own copies
    del bus_df
    gc.collect()
    mem = psutil.virtual_memory()
    print(f"  RAM available after release: {mem.available / 1024**3:.1f} GB")

    # Step 3: Build LightGBM Dataset for finaltrain
    print(f"\n  [3/5] Building LightGBM Dataset...")
    t_ds = time.time()
    finaltrain_set = lgb.Dataset(
        splits["X_finaltrain"],
        label=splits["y_finaltrain"],
        categorical_feature=CATEGORICAL_FEATURES,
        free_raw_data=True,
    )
    finaltrain_set.construct()
    print(f"    Dataset built in {time.time() - t_ds:.1f}s")
    mem = psutil.virtual_memory()
    print(f"    RAM available: {mem.available / 1024**3:.1f} GB")

    # Step 4: Final-train
    print(f"\n  [4/5] Final retraining...")
    hp = task_hyperparams[task]
    best_params = hp["best_params"]
    final_iter = hp["n_estimators_final"]

    params = {
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "seed": LGBM_SEED,
        **best_params,
    }

    print(f"    n_estimators: {final_iter}")
    print(f"    Hyperparameters: {best_params}")

    t_train = time.time()
    final_model = lgb.train(
        params,
        finaltrain_set,
        num_boost_round=final_iter,
        callbacks=[lgb.log_evaluation(period=0)],
    )
    elapsed_train = time.time() - t_train
    print(f"    Final-train complete in {elapsed_train/60:.1f} min ({final_iter} trees)")

    # Release Dataset (model retains its internal representation)
    del finaltrain_set
    gc.collect()

    # Step 5: Predict on 2025
    print(f"\n  [5/5] Predicting on 2025 test set...")
    t_pred = time.time()
    predict_pd = final_model.predict(splits["X_test"], num_iteration=final_iter)
    elapsed_pred = time.time() - t_pred
    print(f"    Prediction complete in {elapsed_pred:.1f}s")
    print(f"    Predict_pd range: [{predict_pd.min():.2f}, {predict_pd.max():.2f}] MW "
          f"(mean: {predict_pd.mean():.2f})")

    # Step 6: Format and write
    print(f"\n  Formatting and writing output parquet...")
    t_write = time.time()
    output_df = format_output(task, predict_pd, splits["test_identity"])
    output_df.to_parquet(forecast_path, index=False, compression="zstd")
    elapsed_write = time.time() - t_write
    size_mb = forecast_path.stat().st_size / 1024**2
    print(f"    Wrote {forecast_path.name}: {len(output_df):,} rows, "
          f"{size_mb:.1f} MB in {elapsed_write:.1f}s")
    print(f"    Sample first row: {output_df.iloc[0].to_dict()}")

    # Release everything for this task
    del splits, final_model, predict_pd, output_df
    gc.collect()
    mem = psutil.virtual_memory()
    elapsed_task = time.time() - t_task
    print(f"\n  ✓ {task} complete in {elapsed_task/60:.1f} min")
    print(f"  RAM available after release: {mem.available / 1024**3:.1f} GB")

    processing_summary[task] = {
        "skipped": False,
        "elapsed_min": elapsed_task / 60,
        "n_estimators": final_iter,
        "weight_features_count": len(weather_in_features),
        "total_features": len(feature_cols),
    }


# ──────────────────────────────────────────────────────────────────────────
# Summary
# ──────────────────────────────────────────────────────────────────────────
elapsed_total = time.time() - t0_outer
print(f"\n{'='*80}")
print(f"All tasks processed in {elapsed_total/60:.1f} min ({elapsed_total/3600:.2f} hr)")
print(f"{'='*80}\n")

print("Per-task summary:")
for task, res in processing_summary.items():
    if res["skipped"]:
        print(f"  {task}: SKIPPED (forecast file already existed)")
    else:
        print(f"  {task}: {res['elapsed_min']:.1f} min, "
              f"n_estimators={res['n_estimators']}, "
              f"total features={res['total_features']} "
              f"(includes {res['weight_features_count']} weather)")

# Verify outputs on disk
print(f"\nFiles on disk:")
for task, path in OUTPUT_PATHS.items():
    if path.exists():
        size_mb = path.stat().st_size / 1024**2
        print(f"  ✓ {path.name}: {size_mb:.1f} MB")
    else:
        print(f"  ✗ MISSING: {path.name}")

mem = psutil.virtual_memory()
print(f"\nFinal RAM state: {mem.available / 1024**3:.1f} GB available / "
      f"{mem.total / 1024**3:.1f} GB total")


[1/2] Processing task: nextday

  [1/5] Loading task data with weather features...
  Loading nextday feature parquets and joining weather...
    Loaded: 130,979,958 rows × 32 cols (12.81 GB) in 10.2s
    Joined weather in 13.5s
    Rows with NaN in any weather col: 14,960 (0.011%)
    Post-join: 130,979,958 rows × 37 cols (15.25 GB)
  RAM available after load: 12.6 GB

  [2/5] Building train/test splits...
    Final-train: 98,552,404 rows × 34 features
    Test:        32,427,554 rows × 34 features
    Weather features included: 5 (['temp_at_hour', 'HDH_at_hour', 'CDH_at_hour', 'temp_trailing_24h_at_fc', 'temp_trailing_168h_at_fc'])
    Total features: 34 (categoricals: ['bus_unique_id', 'zone_name'])
    Build splits in 30.2s
  RAM available after release: 22.6 GB

  [3/5] Building LightGBM Dataset...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may 

### Per-task processing — observations

Notebook 05b completed in **18.5 minutes total** — much faster than the 26-60 minute estimate. The two tasks ran in 9.8 min (nextday) and 8.7 min (nextmonth), with final-train dominating the per-task cost (~6.5 min each) and prediction adding 1-2 min.

**Memory behavior was clean throughout the run.** The kernel restart paid off: peak RAM usage stayed at ~16 GB used (out of 36 GB total) during the heaviest moments. The `free_raw_data=True` parameter on LightGBM Dataset construction released the pandas slices as expected — RAM available rose from 12-13 GB right after data load to 18-22 GB once the Dataset was binned. Between tasks, RAM recovered cleanly to ~25-26 GB via explicit `del` and `gc.collect()`. No macOS memory compression triggered.

**Weather join produced expected NaN counts.** 14,960 rows out of 130,979,958 (0.011%) have NaN in at least one weather column. Decoded: 8 zones × 4 DST spring-forward transitions × ~470 buses present at those zone-hours = ~15,000 rows. This is the same DST artifact we saw at zone level in notebook 04b (32 zone-hours), now expanded to bus-level (each zone-hour has ~470 buses on average that get NaN weather). LightGBM handles these natively via learned missing-value routing.

**Comparison of predict_pd distributions vs notebook 05a (no weather):**

| Metric | 05a nextday | 05b nextday | 05a nextmonth | 05b nextmonth |
|---|---|---|---|---|
| min | -7.49 | -6.70 | 1.91 | 1.92 |
| max | 822.62 | 831.07 | 619.23 | 648.81 |
| mean | 14.12 | 13.90 | 13.70 | 13.67 |

The means barely shift (~0.1-0.2 MW differences between weather and no-weather variants). This is the right qualitative behavior: weather features should change *which* hours and *which* buses get higher/lower predictions, not the aggregate mean across the test set. If weather had moved the mean substantially, the model would be miscalibrated.

**The negative-prediction issue from notebook 05a recurs but is slightly less severe.** Notebook 05a nextday had min=-7.49 MW (0.87% of rows negative); notebook 05b nextday has min=-6.70 MW. We don't have the per-row negative count yet — Cell 4's verification will surface it — but the smaller magnitude suggests weather features helped clamp some of the extrapolation errors. Nextmonth shows no negative predictions in either notebook (the slower learning rate and longer horizon make extrapolation less aggressive).

**Notebook 05b nextmonth max is higher than notebook 05a's (648.81 vs 619.23 MW).** This is the opposite direction from what we might naively expect (more features → smoother predictions). The model now has access to extreme-temperature signal (the WEST 114°F observations from summer 2023, the NOTH 1°F observations from January 2024) and uses that signal to extrapolate to higher peak loads on similar future days. Whether this generalizes to 2025 actuals is for notebook 06 to evaluate.

**One LightGBM warning observed during nextday Dataset construction:**

[LightGBM] [Warning] Categorical features with more bins than the configured maximum
bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be
ignored with a large number of categories.

This warning is benign and was expected. `bus_unique_id` has 4,208 categories, which exceeds LightGBM's default `max_bin=255`. The warning is telling us that LightGBM cannot represent each bus as its own bin — it groups them into 255 bins based on their mean target value (target-mean encoding, the standard LightGBM approach for high-cardinality categoricals). This is the same behavior notebook 05a used; the warning just wasn't visible in 05a's output because of different logging defaults. The grouping is methodologically fine — `cat_smooth=20` provides regularization toward zone-level means for buses with sparse support.

**File sizes:**
- nextday: 130.1 MB (vs notebook 05a's 128.1 MB — almost identical, weather features add maybe 1-2 MB)
- nextmonth: 104.8 MB (vs notebook 05a's 101.5 MB — same pattern)

Both files match the canonical 32,427,554-row count required for evaluation alignment with notebooks 03, 04, 04b, and 05a.

**The 2×2 ablation matrix is now structurally complete on disk:**

| Architecture / Weather | No weather | With weather |
|---|---|---|
| Zone-direct + top-down | `forecast_zone_direct_lgbm_*.parquet` (notebook 04) | `forecast_zone_direct_lgbm_weather_*.parquet` (notebook 04b) |
| Global-bus direct | `forecast_global_bus_lgbm_*.parquet` (notebook 05a) | `forecast_global_bus_lgbm_weather_*.parquet` (notebook 05b) |

Cell 4 will verify the two new forecast files against the assignment schema. After that, notebook 05b is ready to commit, and we move to notebook 06 — where the substantive evaluation actually happens.

## Final verification

Cell 4 mirrors the verification pattern from notebooks 04b and 05a: hard-assertion checks against every constraint the assignment imposes on the output schema. For each of the two forecast files we confirm:

1. **File exists** on disk at the expected path
2. **Schema** is the canonical 7-column format in the exact required order
3. **Row count** is exactly 32,427,554 (matching notebooks 03, 04, 04b, 05a)
4. **Zero NaN** values in `predict_pd`
5. **model_name** is unique within the file and matches `MODEL_NAMES[task]`
6. **forecast_created_at unique count**: 364 for nextday (one per non-excluded day), 12 for nextmonth (one per month)
7. **2025-12-04 systematically excluded** from `target_date`
8. **HE range** is exactly [1, 24]
9. **Bus universe**: 3,953 unique bus_ids (matching the 2025 bus inventory across the pipeline)
10. **predict_pd statistics**: range, mean, and explicit reporting of any negative predictions (we expect a small fraction in nextday based on notebook 05a's behavior; nextmonth should be all non-negative)

If any assertion fails, the cell raises immediately. Passing all checks means the file is ready for notebook 06's evaluation alongside notebook 04, 04b, and 05a's forecasts.

In [7]:
"""
Final verification of the two weather-augmented global-bus forecast files written by Cell 3.

Confirms each file:
  - Exists on disk with expected size
  - Has the canonical 7-column schema in correct order
  - Has the expected 32,427,554 rows
  - Has no NaN values in predict_pd
  - Has the correct unique value count for forecast_created_at
    (364 for nextday, 12 for nextmonth)
  - Excludes 2025-12-04 from target_date
  - Has HE values in the range [1, 24]
  - Bus_id values cover exactly the 3,953-bus 2025 universe
  - Reports any negative predict_pd values explicitly

Runtime: <30 seconds (parquet metadata + light data sampling).
"""

t0 = time.time()

REQUIRED_SCHEMA_ORDER = [
    "model_name", "forecast_created_at", "target_date", "he", "bus_id", "zone_id", "predict_pd"
]

# Expected forecast_created_at unique value counts
EXPECTED_FC_UNIQUE = {"nextday": 364, "nextmonth": 12}
EXPECTED_ROWS = 32_427_554
EXPECTED_BUSES = 3_953

print(f"{'='*70}")
print("Final verification — notebook 05b weather-augmented global-bus forecasts")
print(f"{'='*70}\n")

for task, path in OUTPUT_PATHS.items():
    print(f"--- {task}: {path.name} ---")

    # File exists and size
    assert path.exists(), f"Missing forecast file: {path}"
    size_mb = path.stat().st_size / 1024**2
    print(f"  File size: {size_mb:.1f} MB")

    # Read full file
    df = pq.read_table(path).to_pandas()

    # Schema verification (order + presence)
    assert list(df.columns) == REQUIRED_SCHEMA_ORDER, (
        f"Schema order mismatch. Expected {REQUIRED_SCHEMA_ORDER}, got {list(df.columns)}"
    )
    print(f"  Schema:    ✓ all 7 columns in correct order")

    # Row count
    assert len(df) == EXPECTED_ROWS, (
        f"Got {len(df):,} rows, expected {EXPECTED_ROWS:,}"
    )
    print(f"  Row count: ✓ {len(df):,} rows (matches notebooks 03, 04, 04b, 05a)")

    # No NaN in predict_pd
    n_nan = df["predict_pd"].isna().sum()
    assert n_nan == 0, f"Found {n_nan} NaN values in predict_pd"
    print(f"  NaN check: ✓ 0 NaN values in predict_pd")

    # model_name uniqueness and value
    model_names = df["model_name"].unique()
    expected_model_name = MODEL_NAMES[task]
    assert len(model_names) == 1, f"Multiple model_names found: {model_names}"
    assert model_names[0] == expected_model_name, (
        f"model_name mismatch: expected '{expected_model_name}', got '{model_names[0]}'"
    )
    print(f"  model_name: ✓ '{expected_model_name}' (unique)")

    # forecast_created_at unique count
    n_fc_unique = df["forecast_created_at"].nunique()
    expected_fc = EXPECTED_FC_UNIQUE[task]
    assert n_fc_unique == expected_fc, (
        f"forecast_created_at: got {n_fc_unique} unique values, expected {expected_fc}"
    )
    print(f"  fc_at count: ✓ {n_fc_unique} unique values "
          f"({'one per non-excluded day' if task == 'nextday' else 'one per month'})")

    # 2025-12-04 excluded from target_date
    n_dec4 = (df["target_date"] == pd.Timestamp("2025-12-04")).sum()
    assert n_dec4 == 0, f"2025-12-04 should be excluded from target_date, but found {n_dec4} rows"
    print(f"  Dec 4 check: ✓ 2025-12-04 absent from target_date")

    # HE range
    he_min, he_max = df["he"].min(), df["he"].max()
    assert he_min == 1 and he_max == 24, f"HE range should be [1, 24], got [{he_min}, {he_max}]"
    print(f"  HE range:  ✓ [{he_min}, {he_max}]")

    # Bus universe check
    bus_universe = set(df["bus_id"].unique())
    assert len(bus_universe) == EXPECTED_BUSES, (
        f"Bus count: got {len(bus_universe)}, expected {EXPECTED_BUSES}"
    )
    print(f"  Buses:     ✓ {len(bus_universe):,} unique bus_ids in 2025 predictions")

    # predict_pd statistics
    pred_min, pred_max, pred_mean = df["predict_pd"].min(), df["predict_pd"].max(), df["predict_pd"].mean()
    print(f"  predict_pd: min={pred_min:.2f}, max={pred_max:.2f}, mean={pred_mean:.2f} MW")
    if pred_min < 0:
        n_neg = (df["predict_pd"] < 0).sum()
        pct_neg = 100 * n_neg / len(df)
        print(f"    Note: {n_neg:,} rows ({pct_neg:.2f}%) have negative predict_pd")
        print(f"    (LightGBM extrapolation; notebook 06 will decide on clipping)")
    else:
        print(f"    ✓ All predictions non-negative")

    # Spot check: first and last row
    print(f"  First row: {df.iloc[0].to_dict()}")
    print(f"  Last row:  {df.iloc[-1].to_dict()}")

    # Per-zone coverage
    per_zone = df.groupby("zone_id", observed=True).size()
    print(f"  Per-zone rows: min={per_zone.min():,}, max={per_zone.max():,}, "
          f"std={per_zone.std():.0f}")

    print()
    del df
    gc.collect()

elapsed = time.time() - t0
print(f"{'='*70}")
print(f"✓ All verification checks passed in {elapsed:.1f}s")
print(f"{'='*70}")
print(f"\nWeather-augmented global-bus forecast files ready for notebook 06 evaluation:")
for task, path in OUTPUT_PATHS.items():
    print(f"  {path.name}")

# Verify the 2×2 ablation matrix is complete on disk
print(f"\n{'='*70}")
print("2×2 weather ablation matrix — files on disk:")
print(f"{'='*70}")

ablation_files = {
    "Zone-direct, no weather":     FORECASTS_DIR / "forecast_zone_direct_lgbm_nextday.parquet",
    "Zone-direct, weather":        FORECASTS_DIR / "forecast_zone_direct_lgbm_weather_nextday.parquet",
    "Global-bus, no weather":      FORECASTS_DIR / "forecast_global_bus_lgbm_nextday.parquet",
    "Global-bus, weather":         FORECASTS_DIR / "forecast_global_bus_lgbm_weather_nextday.parquet",
}
for label, path in ablation_files.items():
    if path.exists():
        size_mb = path.stat().st_size / 1024**2
        print(f"  ✓ {label:<25}: {path.name} ({size_mb:.1f} MB)")
    else:
        print(f"  ✗ MISSING: {label:<25}: {path.name}")

ablation_files_nextmonth = {
    "Zone-direct, no weather":     FORECASTS_DIR / "forecast_zone_direct_lgbm_nextmonth.parquet",
    "Zone-direct, weather":        FORECASTS_DIR / "forecast_zone_direct_lgbm_weather_nextmonth.parquet",
    "Global-bus, no weather":      FORECASTS_DIR / "forecast_global_bus_lgbm_nextmonth.parquet",
    "Global-bus, weather":         FORECASTS_DIR / "forecast_global_bus_lgbm_weather_nextmonth.parquet",
}
print(f"\n  Nextmonth task:")
for label, path in ablation_files_nextmonth.items():
    if path.exists():
        size_mb = path.stat().st_size / 1024**2
        print(f"  ✓ {label:<25}: {path.name} ({size_mb:.1f} MB)")
    else:
        print(f"  ✗ MISSING: {label:<25}: {path.name}")

Final verification — notebook 05b weather-augmented global-bus forecasts

--- nextday: forecast_global_bus_lgbm_weather_nextday.parquet ---
  File size: 130.1 MB
  Schema:    ✓ all 7 columns in correct order
  Row count: ✓ 32,427,554 rows (matches notebooks 03, 04, 04b, 05a)
  NaN check: ✓ 0 NaN values in predict_pd
  model_name: ✓ 'global_bus_lgbm_weather_nextday' (unique)
  fc_at count: ✓ 364 unique values (one per non-excluded day)
  Dec 4 check: ✓ 2025-12-04 absent from target_date
  HE range:  ✓ [1, 24]
  Buses:     ✓ 3,953 unique bus_ids in 2025 predictions
  predict_pd: min=-6.70, max=831.07, mean=13.90 MW
    Note: 683,157 rows (2.11%) have negative predict_pd
    (LightGBM extrapolation; notebook 06 will decide on clipping)
  First row: {'model_name': 'global_bus_lgbm_weather_nextday', 'forecast_created_at': Timestamp('2024-12-31 00:00:00'), 'target_date': Timestamp('2025-01-01 00:00:00'), 'he': 1, 'bus_id': '36POD_138KV_1', 'zone_id': 'FWES', 'predict_pd': 24.62433433532715}


### Verification — all checks passed, with one observation worth flagging

Both weather-augmented global-bus forecast files conform to the canonical 7-column schema and pass every assignment-imposed constraint:

| Property | nextday | nextmonth |
|---|---|---|
| File size | 130.1 MB | 104.8 MB |
| Row count | 32,427,554 | 32,427,554 |
| forecast_created_at unique values | 364 | 12 |
| Bus universe | 3,953 | 3,953 |
| predict_pd range | [-6.70, 831.07] MW | [1.92, 648.81] MW |
| predict_pd mean | 13.90 MW | 13.67 MW |
| Negative predictions | 683,157 rows (2.11%) | 0 |

**Negative prediction rate increased from notebook 05a.** Notebook 05a's nextday had 282,415 negative predictions (0.87%); notebook 05b's nextday has 683,157 (2.11%) — a **2.4× increase**. The minimum value barely changed (-7.49 → -6.70), so this isn't a few extreme outliers getting worse; it's a meaningful fraction of buses landing in the slightly-negative range. Three plausible explanations:

1. **Hyperparameter mismatch.** Notebook 05a's hyperparameters were tuned by Optuna for a feature set without weather. The regularization (lambda_l1=7.62, cat_smooth=20 for nextday) is calibrated for that no-weather feature distribution; under the weather-augmented set, the same regularization may be too loose, allowing the model to extrapolate more aggressively into negative territory for low-pd buses. This is the explicit cost of the strict warm-start design (documented in Cell 1 as Decision 1).

2. **Weather feature interaction with low-pd buses.** Industrial buses with near-zero baseline load might be getting nudged into negative territory when the model overweights temperature signal under unusual feature combinations. A bus that normally reads 2-3 MW with low temperature sensitivity might get pushed below zero when the model assigns it a strong negative temperature gradient.

3. **Structural property of tree models with strongly-predictive features.** As predictive accuracy improves on most buses, the residual fitting on edge-case buses can get worse — gradient boosting greedily reduces aggregate loss, and a small amount of negative extrapolation on a few percent of rows can be tolerated by the loss function if it improves overall fit elsewhere.

The 2.11% negative rate is methodologically tolerable but worth documenting. Notebook 06's evaluation will clip at zero before computing point metrics (standard practice — physical pd cannot be negative). The unclipped negative-prediction rate itself becomes a comparative diagnostic across the 2×2 matrix:

| Variant | Negative rate (nextday) |
|---|---|
| Zone-direct + weather (04b) | 0.00% (structurally bounded by disaggregation) |
| Global-bus + no weather (05a) | 0.87% |
| Global-bus + weather (05b) | 2.11% |

This is a real architectural difference: top-down disaggregation produces non-negative bus predictions by construction (zone forecast × non-negative share = non-negative). Global-bus direct can extrapolate freely. Adding weather features increases the global-bus negative rate because the additional feature dimensions give the model more directions in which to extrapolate.

**The 2×2 ablation matrix is structurally complete on disk** — all 8 forecast files (4 model variants × 2 tasks) present at canonical paths:

| Architecture / Weather | No weather | With weather |
|---|---|---|
| Zone-direct + top-down | `forecast_zone_direct_lgbm_*.parquet` (141.2 / 120.5 MB) | `forecast_zone_direct_lgbm_weather_*.parquet` (124.3 / 123.1 MB) |
| Global-bus direct | `forecast_global_bus_lgbm_*.parquet` (128.1 / 101.5 MB) | `forecast_global_bus_lgbm_weather_*.parquet` (130.1 / 104.8 MB) |

**Notebook 05b is complete.** Cell 5's commit step puts the notebook on GitHub. After that, the modeling phase of the assignment is fully done — every forecast file required for notebook 06's evaluation is on disk and verified.